In [ ]:
import scipy as sp
import numpy as np
import thewalrus as wr
import itertools
from tqdm.auto import tqdm
import bqplot.pyplot as plt
import bqplot as bq
import ipywidgets as widgets

In [ ]:
def T(n):
    """Defines conversion matrix for thewalrus
    T.T@sigma@T is our converted matrix
    T@sigma@T.T converts back
    """
    v1 = np.array([[1,0]])
    v2 = np.array([[0,1]])
    T1 = sp.linalg.block_diag(*([v1]*n))
    T2 = sp.linalg.block_diag(*([v2]*n))
    T = np.block([[T1],[T2]])
    return T
def sigma(η,ns,nb):
    return np.array([
        [1 +2*η*ns + 2*nb,0,-2*np.sqrt(ns*(1+ns)*η),0],
        [0,1 +2*η*ns + 2*nb,0,2*np.sqrt(ns*(1+ns)*η)],
        [-2*np.sqrt(ns*(1+ns)*η),0,1 +2*ns,0],
        [0,2*np.sqrt(ns*(1+ns)*η),0,1 +2*ns]
    ])
#def cov(η,ns,nb):
#    return T(2)@σ(η,ns,nb)@T(2)
#@functools.cache
#def Ps_pn(η,ns,nb,maxval=10):
#    means = np.zeros(4)
#    covval = cov(η,ns,nb)
#    Ps = wr.quantum.probabilities(means,covval,maxval,atol=1e-10,rtol=1e-7)
#    return Ps
#def Ps_pnij(η,ns,nb,i,j,maxval=10):
#    return Ps_pn(η,ns,nb,maxval)[i][j]
#vec_Ps_pn = np.vectorize(Ps_pnij)
#def to_diff_pn(ηs,*args):
#    ns = args[0]
#    nb = args[1]
#    i = args[2]
#    j = args[3]
#    maxval=args[4]
#    return vec_Ps_pn(ηs,ns,nb,i,j,maxval)
#def FI_pn(ηs,nss,nbs,maxval=10):
#    indices = np.arange(maxval)
#    ηgrid,nsgrid,nbgrid,igrid,jgrid = np.meshgrid(ηs,nss,nbs,indices,indices,indexing='ij')
#    Ps = vec_Ps_pn(ηgrid,nsgrid,nbgrid,igrid,jgrid,maxval)
#    ds = np.zeros((len(ηs),len(nss),len(nbs),maxval,maxval))
#    for i in range(maxval):
#        for j in range(maxval):
#            for k,ns in enumerate(nss):
#                for l,nb in enumerate(nbs):
#                    derivres = sp.differentiate.derivative(to_diff_pn,ηvals,args = [ns,nb,i,j,maxval],initial_step=1e-5)
#                    ds[:,k,l,i,j] = derivres.df
#    presum = ds**2/Ps
#    presum[~np.isfinite(presum)] = 0
#    FI = np.sum(presum,axis=(3,4))
#    return FI


def sigmasu(η,ns,nb):
    bkrd = 1+2*ns*((1+ns)*(1+np.sqrt(η))**2+nb)
    corr = 2*np.sqrt(ns*(1+ns))*(1+np.sqrt(η)+nb+ns*(1+np.sqrt(η))**2)
    return np.array([
        [bkrd + 2*nb,0,-corr,0],
        [0,bkrd+ 2*nb,0,corr],
        [-corr,0,bkrd+2*ns*(1-η),0],
        [0,corr,0,bkrd+2*ns*(1-η)]
    ])
#def covsu(η,ns,nb):
#    return T(2)@σsu(η,ns,nb)@T(2)
#@functools.cache
#def Ps_su(η,ns,nb,maxval=10):
#    means = np.zeros(4)
#    covval = covsu(η,ns,nb)
#    Ps = wr.quantum.probabilities(means,covval,maxval,atol=1e-10,rtol=1e-7)
#    return Ps
#def Ps_suij(η,ns,nb,i,j,maxval=10):
#    return Ps_su(η,ns,nb,maxval)[i][j]
#
#vec_Ps_su = np.vectorize(Ps_suij)
#
#
#def to_diff_su(ηs,*args):
#    ns = args[0]
#    nb = args[1]
#    i = args[2]
#    j = args[3]
#    maxval = args[4]
#    return vec_Ps_su(ηs,ns,nb,i,j,maxval)
#def FI_su(ηs,nss,nbs,maxval=10):
#    indices = np.arange(maxval)
#    ηgrid,nsgrid,nbgrid,igrid,jgrid = np.meshgrid(ηs,nss,nbs,indices,indices,indexing='ij')
#    Ps = vec_Ps_su(ηgrid,nsgrid,nbgrid,igrid,jgrid,maxval)
#    ds = np.zeros((len(ηs),len(nss),len(nbs),maxval,maxval))
#    for i in range(maxval):
#        for j in range(maxval):
#            for k,ns in enumerate(nss):
#                for l,nb in enumerate(nbs):
#                    derivres = sp.differentiate.derivative(to_diff_su,ηvals,args = [ns,nb,i,j,maxval],initial_step=1e-5)
#                    ds[:,k,l,i,j] = derivres.df
#    presum = ds**2/Ps
#    presum[~np.isfinite(presum)] = 0
#    FI = np.sum(presum,axis=(3,4))
#    return FI
#
#
#
#def Ps_pn_jac(η,*args):
#    ns,nb = args[0,1]
#    maxval = args[2]
#    means = np.zeros(4)
#    covval = cov(η,ns,nb)
#    Ps = wr.quantum.probabilities(means,covval,maxval,atol=1e-10,rtol=1e-7)
#    return Ps
#def vec_Ps_pn_jac_gen(*args):
#    def vec_Ps_pn_jac(x):
#        return np.apply_along_axis(Ps_pn_jac, axis=0, arr=x,args = args)
#    return vec_Ps_pn_jac
#def FI_pn_jac(ηs,nss,nbs,maxval=10):
#    indices = np.arange(maxval)
#    ηgrid,nsgrid,nbgrid,igrid,jgrid = np.meshgrid(ηs,nss,nbs,indices,indices,indexing='ij')
#    Ps = vec_Ps_pn(ηgrid,nsgrid,nbgrid,igrid,jgrid,maxval)
#    ds = np.zeros((len(ηs),len(nss),len(nbs),maxval,maxval))
#    for i,ns in enumerate(nss):
#        for j,nb in enumerate(nbs):
#            vec_Ps_pn_jac = vec_Ps_pn_jac_gen(ns,nb,maxval)
#            derivres = sp.differentiate.jacobian(vec_Ps_pn_jac,ηs,initial_step=1e-5)
#            ds[:,i,j,:,:] = derivres,df
#    presum = ds**2/Ps
#    presum[~np.isfinite(presum)] = 0
#    FI = np.sum(presum,axis=(3,4))
#    return FI    
#def Ps_su_jac(η,*args):
#    ns,nb = args[0:2]
#    maxval = args[2]
#    means = np.zeros(4)
#    covval = covsu(η,ns,nb)
#    Ps = wr.quantum.probabilities(means,covval,maxval,atol=1e-10,rtol=1e-7)
#    return Ps
#def vec_Ps_su_jac_gen(*args):
#    maxval: int = args[2]
#    def vec_Ps_su_jac(x):
#        xvals = x[0]
#        output = np.zeros((maxval,maxval,*xvals.shape))
#        for i,val in np.ndenumerate(xvals):
#            output[:,:,*i] = Ps_su_jac(val,*args)
#        return output
#    return vec_Ps_su_jac
#def FI_su_jac(ηs,nss,nbs,maxval=10):
#    indices = np.arange(maxval)
#    ηgrid,nsgrid,nbgrid,igrid,jgrid = np.meshgrid(ηs,nss,nbs,indices,indices,indexing='ij')
#    ds = np.zeros((len(nss),len(nbs),maxval,maxval,len(ηs)))
#    for (i,ns),(j,nb) in tqdm(itertools.product(enumerate(nss),enumerate(nbs)),total=len(nss)*len(nbs),smoothing=.01):
#        vec_ps_su_jac = vec_ps_su_jac_gen(ns,nb,maxval)
#        derivres = sp.differentiate.jacobian(vec_ps_su_jac,ηs,initial_step=1e-5)
#        ds[i,j,:,:,:] = derivres.df
#    ps = vec_ps_su(ηgrid,nsgrid,nbgrid,igrid,jgrid,maxval)
#    dsadj = np.moveaxis(ds,[0,1,2,3,4],[1,2,3,4,0])
#    presum = dsadj**2/ps
#    presum[~np.isfinite(presum)] = 0
#    fi = np.sum(presum,axis=(3,4))
#    return fi
def vec_Ps_generic_jac_gen(cov,ns,nb,maxval):
    means = np.zeros(4)
    T2 = T(2)
    def vec_Ps_su_jac(x):
        xvals = x[0]
        output = np.zeros((maxval,maxval,*xvals.shape))
        for i,val in np.ndenumerate(xvals):
            covval = T2@cov(val,ns,nb)@T2
            Ps = wr.quantum.probabilities(means,covval,maxval)
            output[:,:,*i] = Ps
        return output
    return vec_Ps_su_jac
def FI_generic(etas,nss,nbs,cov,maxval=10):
    indices = np.arange(maxval)
    etagrid,nsgrid,nbgrid = np.meshgrid(etas,nss,nbs,indexing='ij')
    ds = np.zeros((len(nss),len(nbs),maxval,maxval,len(etas)))
    for (i,ns),(j,nb) in tqdm(itertools.product(enumerate(nss),enumerate(nbs)),total=len(nss)*len(nbs),smoothing=.01):
        vec_ps_su_jac = vec_Ps_generic_jac_gen(cov,ns,nb,maxval)
        derivres = sp.differentiate.jacobian(vec_ps_su_jac,etas,initial_step=1e-5)
        ds[i,j,:,:,:] = derivres.df
    T2 = T(2)
    means = np.zeros(4)
    ps = np.zeros((len(nss),len(nbs),maxval,maxval,len(etas)))
    for (i,ns),(j,nb),(k,eta) in tqdm(itertools.product(enumerate(nss),enumerate(nbs),enumerate(etas)),total=len(nss)*len(nbs)*len(etas),smoothing=.01):
        covval = T2@cov(eta,ns,nb)@T2
        Ps = wr.quantum.probabilities(means,covval,maxval)
        ps[i,j,:,:,k] = Ps
    presum = ds**2/ps
    presum[~np.isfinite(presum)] = 0
    fi = np.sum(presum,axis=(2,3))
    return fi


In [ ]:
Neta = 100
Ns = 5
Nb = 5
etavals = np.linspace(1e-3,1-1e-3,Neta,dtype=np.double)
nsvals = np.logspace(-4,.5,Ns,dtype=np.double)
nbvals = np.insert(np.logspace(-3,-.5,Nb-1,dtype=np.double),0,0)
maxval = 7

In [ ]:
#FIspn = FI_pn(ηvals,nsvals,nbvals,maxval)
#FIssu = FI_su_jac(ηvals,nsvals,nbvals,maxval)
FIspn = FI_generic(etavals,nsvals,nbvals,sigma,maxval)
FIssu= FI_generic(etavals,nsvals,nbvals,sigmasu,maxval)


In [ ]:
nsslider = widgets.IntSlider(
    value=0,
    min=0,
    max=Ns-1,
    step=1,
    description='Ns index',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
nbslider = widgets.IntSlider(
    value=0,
    min=0,
    max=Nb-1,
    step=1,
    description='Nb index:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
fig = plt.figure()
curve = plt.plot(etavals,FIssu[0,0,:]-FIspn[0,0,:])
plt.xlabel("Loss Parameter (η)")
plt.ylabel("FI Difference")
def display_graph(nsi,nbi):
    curve.y=FIssu[nsi,nbi,:]-FIspn[nsi,nbi,:]
    fig.title = f"FI Difference for ns {nsvals[nsi]} and nb {nbvals[nbi]}"


In [ ]:
widgets.interact(display_graph,nsi = nsslider,nbi = nbslider,continuous=False)
fig